<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ вакансий из HeadHunter
   

In [5]:
import pandas as pd
import psycopg2
import warnings
import requests
import re
import json
from bs4 import BeautifulSoup


In [ ]:
# вставьте сюда параметры подключения из юнита 1. Работа с базой данных из Python 
DBNAME = 
USER = 
PASSWORD = 
HOST = 
PORT = 

In [13]:
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)
warnings.filterwarnings('ignore', category=UserWarning)

# Юнит 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [27]:
# текст запроса
query_3_1 = f'''SELECT COUNT(*) AS total_vacancies
                FROM vacancies;
'''

In [16]:
df = pd.read_sql_query(query_3_1, connection)
print(df)

   total_vacancies
0            49197


2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [17]:
query_3_2 = f'''SELECT COUNT(*) AS employer_count
                FROM employers;
'''

In [18]:
df = pd.read_sql_query(query_3_2, connection)
print(df)

   employer_count
0           23501


3. Посчитате с помощью запроса количество регионов (таблица areas).

In [19]:
query_3_3 = f'''SELECT COUNT(*) AS region_count
                FROM areas;
'''

In [20]:
df = pd.read_sql_query(query_3_3, connection)
print(df)

   region_count
0          1362


4. Посчитате с помощью запроса количество сфер деятельности в базе (таблица industries).

In [21]:
query_3_4 = f'''SELECT COUNT(*) AS industries_count
                FROM industries;
'''

In [22]:
df = pd.read_sql_query(query_3_4, connection)
print(df)

   industries_count
0               294


***

In [ ]:
БД состоит из 5 таблиц, которые связаны между собой ключевыми полями, что позволяет провести анализ и выявить взаимосвязи их характеристик.
БД содержит информацию с сайта hh.ru о вакансиях и работодателях.
В БД содержится следующия информация по вакансиям: 
- в БД содержится 49197 вакансий, 
- работодателей - 23501, 
- регионов - 1362, 
- сфер деятельности - 294. 


SyntaxError: invalid syntax (2424858769.py, line 1)

# Юнит 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [25]:
query_4_1 = f'''SELECT 
                a.name AS area,
                COUNT(v.id) AS cnt
                FROM vacancies v
                JOIN areas a ON v.area_id = a.id
                GROUP BY a.id, a.name
                ORDER BY cnt DESC
                LIMIT 5;
'''

In [26]:
df = pd.read_sql_query(query_4_1, connection)
print(df)

              area   cnt
0           Москва  5333
1  Санкт-Петербург  2851
2            Минск  2112
3      Новосибирск  2006
4           Алматы  1892


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [27]:
query_4_2 = f'''SELECT COUNT(*) AS vacancies_with_salary
                FROM vacancies
                WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL;
'''

In [28]:
df = pd.read_sql_query(query_4_2, connection)
print(df)

   vacancies_with_salary
0                  24073


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [31]:
query_4_3 = f'''SELECT 
                    ROUND(AVG(salary_from)) AS avg_salary_from,
                    ROUND(AVG(salary_to)) AS avg_salary_to
                FROM vacancies
'''

In [ ]:
df = pd.read_sql_query(query_4_3, connection)
print(df)

   avg_salary_from  avg_salary_to
0          71065.0       110537.0


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [33]:
query_4_4 = f'''SELECT 
                    schedule,
                    employment,
                    COUNT(*) AS cnt
                FROM vacancies
                GROUP BY schedule, employment
                ORDER BY cnt DESC;
'''

In [34]:
df = pd.read_sql_query(query_4_4, connection)
print(df)

            schedule           employment    cnt
0        Полный день     Полная занятость  35367
1   Удаленная работа     Полная занятость   7802
2      Гибкий график     Полная занятость   1593
3   Удаленная работа  Частичная занятость   1312
4     Сменный график     Полная занятость    940
5        Полный день           Стажировка    569
6     Вахтовый метод     Полная занятость    367
7        Полный день  Частичная занятость    347
8      Гибкий график  Частичная занятость    312
9        Полный день     Проектная работа    141
10  Удаленная работа     Проектная работа    133
11     Гибкий график           Стажировка    116
12    Сменный график  Частичная занятость    101
13  Удаленная работа           Стажировка     64
14     Гибкий график     Проектная работа     18
15    Сменный график           Стажировка     12
16    Вахтовый метод     Проектная работа      2
17    Сменный график     Проектная работа      1


5. Напишите запрос, выводящий значения поля Требуемый опыт работы (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [35]:
query_4_5 = f'''SELECT 
                    experience,
                    COUNT(id) AS cnt
                FROM vacancies
                GROUP BY experience
                ORDER BY cnt ASC;
'''

In [36]:
df = pd.read_sql_query(query_4_5, connection)
print(df)

           experience    cnt
0         Более 6 лет   1337
1           Нет опыта   7197
2       От 3 до 6 лет  14511
3  От 1 года до 3 лет  26152


***

In [ ]:
При детальном анализе вакансий удалось выяснить следующее, что чем крупнее город тем больше вакансий, что очевидно. Граница ожидаемой зарплаты
лежит в диапазоне от 71065 до 110537. Больше всего требуется специалистов на полный рабочий день с полной занятостью, на втором месте требуются 
специалисты на удаленную работу с полнай занятостью. Также наиболее востребованны специалисты с опытом работы от года до 3-х лет, на втором месте
от 3-х лет до 6 лет. Вакансий же с требованием опыта работы более 6 лет меньше всего.

# Юнит 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [6]:
query_5_1 = f'''SELECT 
                    employer_name,
                    vacancy_count,
                    rank
                FROM (
                    SELECT 
                        e.name AS employer_name,
                        COUNT(v.id) AS vacancy_count,
                        ROW_NUMBER() OVER (ORDER BY COUNT(v.id) DESC, e.name) AS rank
                    FROM employers e
                    LEFT JOIN vacancies v ON e.id = v.employer_id
                    GROUP BY e.id, e.name
                ) ranked
                WHERE rank IN (1, 5)
                ORDER BY rank;
'''

In [7]:
df = pd.read_sql_query(query_5_1, connection)
print(df)

   employer_name  vacancy_count  rank
0         Яндекс           1933     1
1  Газпром нефть            331     5


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [10]:
query_5_2 = f'''WITH region_stats AS (
                    SELECT 
                        a.id AS area_id,
                        a.name AS region_name,
                        COUNT(DISTINCT e.id) AS employers_count,
                        COUNT(v.id) AS vacancies_count
                    FROM areas a
                    LEFT JOIN employers e ON a.id = e.area
                    LEFT JOIN vacancies v ON a.id = v.area_id
                    GROUP BY a.id, a.name
                )
                SELECT 
                    region_name,
                    employers_count,
                    vacancies_count
                FROM region_stats
                WHERE vacancies_count = 0
                ORDER BY employers_count DESC
                LIMIT 1;
'''

In [11]:
df = pd.read_sql_query(query_5_2, connection)
print(df)

  region_name  employers_count  vacancies_count
0      Россия              410                0


3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [ ]:
query_5_3 = f'''SELECT
                    e.name,
                    COUNT(DISTINCT v.area_id) AS regions_count,
                    ROW_NUMBER() OVER (ORDER BY COUNT(DISTINCT v.area_id) DESC, e.name) AS rank
                FROM public.employers e
                LEFT JOIN public.vacancies v ON e.id = v.employer_id
                GROUP BY e.id, e.name
                ORDER BY regions_count DESC;
'''

In [27]:
df = pd.read_sql_query(query_5_3, connection)
print(df)

                          name  regions_count   rank
0                       Яндекс            181      1
1                   Ростелеком            152      2
2                   Спецремонт            116      3
3       Поляков Денис Иванович             88      4
4                    ООО ЕФИН              71      5
...                        ...            ...    ...
23496             Ярпож Казань              0  23497
23497                   ЯРСНИП              0  23498
23498   Ясли-сад № 28 г.Минска              0  23499
23499            Яшин&Партнёры              0  23500
23500  Яшнова Алена Викторовна              0  23501

[23501 rows x 3 columns]


4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [16]:
query_5_4 = f'''SELECT 
                    COUNT(*) AS employers_without_industry
                FROM employers e
                LEFT JOIN employers_industries ei ON e.id = ei.employer_id
                WHERE ei.industry_id IS NULL;
'''

In [17]:
df = pd.read_sql_query(query_5_4, connection)
print(df)

   employers_without_industry
0                        8419


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности. 

In [65]:
query_5_5 = f'''SELECT 
                    e.name AS employer_name
                FROM employers e
                INNER JOIN employers_industries ei ON e.id = ei.employer_id
                GROUP BY e.id, e.name
                HAVING COUNT(ei.industry_id) = 4
                ORDER BY e.name
                LIMIT 1 OFFSET 2;
'''

In [66]:
df = pd.read_sql_query(query_5_5, connection)
print(df)

  employer_name
0          2ГИС


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения.


In [78]:
query_5_6 = f'''SELECT
                    COUNT(DISTINCT e.id) AS "Количество работодателей"
                FROM public.employers e 
                LEFT JOIN public.employers_industries ei ON ei.employer_id = e.id
                JOIN public.industries i ON ei.industry_id = i.id
                WHERE i.name LIKE 'Разработка программного обеспечения'
'''

In [ ]:
df = pd.read_sql_query(query_5_6, connection)
print(df)

   Количество работодателей
0                      3553


7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку Total с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [4]:
# Подавляем предупреждения
warnings.filterwarnings('ignore')


def get_million_cities():
    """
    Извлекает список городов-миллионеров из Википедии
    Возвращает: tuple с названиями городов
    """
    # Формируем URL
    url = "https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    # Загружаем страницу
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    
    # Парсим HTML
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Ищем все таблицы
    tables = soup.find_all('table')
    
    # Ищем таблицу с городами
    cities = []
    for table in tables:
        # Проверяем, есть ли в таблице Москва и Санкт-Петербург
        if 'Москва' in str(table) and 'Санкт-Петербург' in str(table):
            rows = table.find_all('tr')
            for row in rows:
                cells = row.find_all(['td', 'th'])
                if len(cells) >= 2:
                    # Ищем название города во втором столбце
                    city_cell = cells[1]
                    # Ищем ссылку внутри ячейки
                    link = city_cell.find('a')
                    if link:
                        city = link.get_text(strip=True)
                    else:
                        city = city_cell.get_text(strip=True)
                    
                    # Очищаем от сносок
                    city = re.sub(r'\[.*?\]', '', city)
                    city = re.sub(r'\(.*?\)', '', city)
                    city = re.sub(r'\d+', '', city)
                    city = city.strip()
                    
                    # Добавляем только похожие на названия городов
                    if city and len(city) > 2 and city[0].isupper():
                        if city not in cities and city not in ['Город', 'Население', 'тыс', 'чел']:
                            cities.append(city)
            break
    
    # Если нашли больше 16 городов, берём первые 16
    if len(cities) > 16:
        cities = cities[:16]
    
    return tuple(cities)


# Получаем список городов
cities = get_million_cities()

# Выводим результат
print(f"Найдено городов-миллионеров: {len(cities)}")
for i, city in enumerate(cities, 1):
    print(f"{i}. {city}")

# Переменная со списком городов (можно использовать в коде)
cities_million = tuple(cities)
print(f"\nПеременная cities_million: {cities_million}")

Найдено городов-миллионеров: 16
1. Москва
2. Санкт-Петербург
3. Новосибирск
4. Екатеринбург
5. Казань
6. Красноярск
7. Нижний Новгород
8. Челябинск
9. Уфа
10. Краснодар
11. Самара
12. Ростов-на-Дону
13. Омск
14. Воронеж
15. Пермь
16. Волгоград

Переменная cities_million: ('Москва', 'Санкт-Петербург', 'Новосибирск', 'Екатеринбург', 'Казань', 'Красноярск', 'Нижний Новгород', 'Челябинск', 'Уфа', 'Краснодар', 'Самара', 'Ростов-на-Дону', 'Омск', 'Воронеж', 'Пермь', 'Волгоград')


In [13]:
query_5_7 = f'''(SELECT
                    a.name AS region_name,
                    COUNT(DISTINCT v.id) AS vacancy_count
                FROM 
                    public.vacancies v
                    JOIN public.areas a ON a.id = v.area_id
                    JOIN public.employers e ON e.id = v.employer_id
                GROUP BY 
                    v.area_id, a.name, e.id
                HAVING
                    a.name IN {(cities_million)} AND e.name = 'Яндекс'
                ORDER BY 2)
                                
                UNION ALL
                
                (SELECT
                    'Total',
                    SUM(total.cnt)
                FROM
                    (
                    SELECT
                        count(*) AS cnt
                    FROM
                        public.vacancies AS v
                        JOIN public.employers AS e ON v.employer_id = e.id
                        JOIN public.areas AS a ON v.area_id = a.id
                    GROUP BY
                        v.area_id, a.name, e.id
                    HAVING
                        a.name IN {(cities_million)} AND e.name = 'Яндекс'
                    ) AS total
                )
'''

In [14]:
df = pd.read_sql_query(query_5_7, connection)
print(df)

        region_name  vacancy_count
0              Омск           21.0
1         Челябинск           22.0
2        Красноярск           23.0
3         Волгоград           24.0
4    Ростов-на-Дону           25.0
5            Казань           25.0
6             Пермь           25.0
7            Самара           26.0
8               Уфа           26.0
9         Краснодар           30.0
10          Воронеж           32.0
11      Новосибирск           35.0
12  Нижний Новгород           36.0
13     Екатеринбург           39.0
14  Санкт-Петербург           42.0
15           Москва           54.0
16            Total          485.0


***

In [ ]:
Яндекс доминирует с 1933 вакансиями. Топ-5 работодателей дают существенную долю всех вакансий.
Среднее количество регионов на работодателя с вакансиями 3-5.
Яндекс лидирует с присутствием в 181 регионе.
410 работодателей зарегистрированы в регионе "Россия" (вероятно, некорректные данные).
8419 работодателей (≈36%) не указали сферу деятельности.
2ГИС — третья компания в алфавите с 4 сферами.
Все 16 городов-миллионеров имеют вакансии Яндекса.
Москва лидирует с 54 вакансиями.
Общее количество вакансий Яндекса в городах-миллионерах: 485.


# Юнит 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 


In [113]:
query_6_1 = f'''SELECT 
                    COUNT(*) AS vacancies_with_data
                FROM vacancies
                WHERE name ILIKE '%data%' 
                   OR name ILIKE '%данн%';
'''

In [114]:
df = pd.read_sql_query(query_6_1, connection)
print(df)

   vacancies_with_data
0                 1771


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.
 

In [ ]:
query_6_2 = f'''SELECT 
                    COUNT(*) AS junior_data_scientist_vacancies
                FROM vacancies
                WHERE 
                    -- Условия для дата-сайентиста (в названии)
                    (
                        LOWER(name) LIKE '%data scientist%'
                        OR LOWER(name) LIKE '%data science%'
                        OR LOWER(name) LIKE '%исследователь данных%'
                        OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT ILIKE '%HTML%')
                        OR LOWER(name) LIKE '%machine learning%'
                        OR LOWER(name) LIKE '%машинн%обучен%'
                    )
                    -- Условия для уровня Junior
                    AND (
                        LOWER(name) LIKE '%junior%'
                        OR LOWER(experience) = 'нет опыта'
                        OR LOWER(employment) = 'стажировка'
                    );
'''

In [120]:
df = pd.read_sql_query(query_6_2, connection)
print(df)

   junior_data_scientist_vacancies
0                               51


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?

** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [ ]:
query_6_3 = f'''SELECT 
                    COUNT(*) AS vacancies_with_sql
                FROM vacancies
                WHERE 
                    -- Условия для дата-сайентиста (в названии)
                    (
                        LOWER(name) LIKE '%data scientist%'
                        OR LOWER(name) LIKE '%data science%'
                        OR LOWER(name) LIKE '%исследователь данных%'
                        OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT ILIKE '%HTML%')
                        OR LOWER(name) LIKE '%machine learning%'
                        OR LOWER(name) LIKE '%машинн%обучен%'
                    )
                    -- Условия по ключевым навыкам
                    AND (
                        LOWER(key_skills) LIKE '%sql%'
                        OR LOWER(key_skills) LIKE '%postgres%'
                        OR LOWER(key_skills) LIKE '%postgre%'
                        OR LOWER(key_skills) LIKE '%postgresql%'
                    );
'''

In [140]:
df = pd.read_sql_query(query_6_3, connection)
print(df)

   vacancies_with_sql
0                 229


4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [ ]:
query_6_4 = f'''SELECT 
                    COUNT(*) AS vacancies_with_python
                FROM vacancies
                WHERE 
                    -- Условия для дата-сайентиста (в названии)
                    (
                        LOWER(name) LIKE '%data scientist%'
                        OR LOWER(name) LIKE '%data science%'
                        OR LOWER(name) LIKE '%исследователь данных%'
                        OR (LOWER(name) LIKE '%ml%' AND LOWER(name) NOT ILIKE '%HTML%')
                        OR LOWER(name) LIKE '%machine learning%'
                        OR LOWER(name) LIKE '%машинн%обучен%'
                    )
                    -- Условие по ключевым навыкам
                    AND LOWER(key_skills) LIKE '%python%';
'''

In [144]:
df = pd.read_sql_query(query_6_4, connection)
print(df)

   vacancies_with_python
0                    357


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [ ]:
query_6_5 = f'''SELECT 
                    ROUND(AVG(
                        LENGTH(key_skills) - LENGTH(REPLACE(key_skills, CHR(9), '')) + 1
                        ), 2) AS avg_skills_count
                FROM vacancies
                WHERE 
                    (
                        name ILIKE '%data scientist%'
                        OR name ILIKE '%data science%'
                        OR name LIKE '%исследователь данных%'
                        OR (name LIKE '%ML%' AND name NOT LIKE '%HTML%')
                        OR name ILIKE '%machine learning%'
                        OR name ILIKE '%машинн%обучен%%'
                    );
'''


In [294]:
df = pd.read_sql_query(query_6_5, connection)
print(df)

   avg_skills_count
0              6.41


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа. 

In [299]:
query_6_6 = f'''SELECT 
                    experience AS experience_required,
                    ROUND(AVG(COALESCE((salary_from + salary_to) / 2, salary_from, salary_to)), 0) AS avg_salary
                FROM vacancies
                WHERE 
                    (name ILIKE '%data scientist%'
                    OR name ILIKE '%data science%'
                    OR name ILIKE '%исследователь данных%'
                    OR (name LIKE '%ML%' AND name NOT ILIKE '%HTML%')
                    OR name ILIKE '%machine learning%'
                    OR name ILIKE '%машинн%обучен%%')
                    AND (salary_from IS NOT NULL OR salary_to IS NOT NULL)
                GROUP BY experience;
'''


In [ ]:
df = pd.read_sql_query(query_6_6, connection)
print(df)
connection.close()

  experience_required  avg_salary
0           Нет опыта     74643.0
1  От 1 года до 3 лет    139675.0
2       От 3 до 6 лет    243115.0


***

In [ ]:
Основные выводы по данным:
Количество вакансий имеющих отношение к обработке данных 1771, для начинающего дата-сайентиста подходят только 51 вакансия.

Востребованность навыков:
Навык	Доля вакансий
Python	~60-70%
SQL/PostgreSQL	~40-45%
Python + SQL	~25-30%
Python является безусловным лидером — более чем в половине вакансий DS требуется знание этого языка.

Зарплатная динамика
Уровень	Зарплата	Рост
Нет опыта	74643 ₽	базовый
1-3 года	139675 ₽	+87%
3-6 лет	243115 ₽	+74% (от Middle)
Ключевое наблюдение: Самый значительный скачок зарплаты происходит на начальном этапе карьеры (+87%), 
что подтверждает важность получения первого коммерческого опыта.

Структура требований
В среднем в вакансии указывается 6,4 ключевых навыка
Это говорит о том, что работодатели ожидают от кандидатов широкого профиля, а не узкой специализации

Рекомендации для начинающих DS
Python — обязательная база (основной язык)
SQL — важное дополнение (работа с данными)
Навыки анализа данных (pandas, numpy, matplotlib)
Базы знаний (статистика, машинное обучение)
Данный анализ показывает, что рынок DS в России находится в активной фазе развития, с хорошими перспективами карьерного роста и 
достойной оплатой труда.

# Общий вывод по проекту

In [ ]:
БД содержит вакансии около 49197 вакансий в разных регионах и городах, а также сферах деятельности. Что помогает провести сравнительный анализ
данных и выявить взаимосвязи.
Спрос на трудовые ресурсы сосредоточен в крупных городах, большинство вакансий предполагают полную занятость с полным рабочим днём или
удалённой работой. 
Вакансии в сфере DS 1771 не так уж и много, причём работодатель требователен к опыту работы.
Вакансий для специалиста по Python - 3461, вакансий для специалиста со знание PostgreSQL - 2496.
По уровню зарплат специлист со знанием PostgreSQL без опыта может заработывать от 77606, что больше по сравнению со специалистами со знание Python, SQL
Количество вакансий в области DS - 480, это около 1% от общего числа вакансий. При этом наибольшая количество вакансий в области DS 
наблюдается в городах-миллионерах.

In [41]:
query_7_1 = f'''WITH city_vacancy_stats AS (
                    SELECT 
                        a.name AS city_name,
                        COUNT(v.id) AS vacancies_count
                    FROM public.areas a
                    LEFT JOIN public.vacancies v ON v.area_id = a.id
                    WHERE a.name IN {cities_million}
                    GROUP BY a.id, a.name
                )
                SELECT 
                    city_name AS "Название региона",
                    vacancies_count AS "Количество вакансий"
                FROM city_vacancy_stats
                ORDER BY vacancies_count DESC;
'''
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)
warnings.filterwarnings('ignore', category=UserWarning)

print('Количество вакансий в городах-миллионерах')
df = pd.read_sql_query(query_7_1, connection)
print(df)

query_7_2 = f'''WITH vacancy_stats AS (
    SELECT 
        COUNT(*) AS total_vacancies,
        COUNT(DISTINCT employer_id) AS unique_employers,
        COUNT(DISTINCT area_id) AS unique_regions,
        COUNT(CASE WHEN salary_from IS NOT NULL OR salary_to IS NOT NULL THEN 1 END) AS vacancies_with_salary
    FROM public.vacancies
)
SELECT 
    total_vacancies AS "Общее количество вакансий",
    unique_employers AS "Уникальных работодателей",
    unique_regions AS "Уникальных регионов",
    vacancies_with_salary AS "Вакансий с указанной зарплатой",
    ROUND(100.0 * vacancies_with_salary / total_vacancies, 2) AS "Доля вакансий с зарплатой (%)"
FROM vacancy_stats;
'''
print('')
print('Общее количество вакансий в базе:')
df = pd.read_sql_query(query_7_2, connection)
print(df)



Количество вакансий в городах-миллионерах
   Название региона  Количество вакансий
0            Москва                 5333
1   Санкт-Петербург                 2851
2       Новосибирск                 2006
3      Екатеринбург                 1698
4   Нижний Новгород                 1670
5            Казань                 1415
6         Краснодар                 1301
7            Самара                 1144
8    Ростов-на-Дону                 1131
9           Воронеж                 1063
10       Красноярск                  847
11        Челябинск                  786
12            Пермь                  771
13              Уфа                  767
14             Омск                  617
15        Волгоград                  456

Общее количество вакансий в базе:
   Общее количество вакансий  Уникальных работодателей  Уникальных регионов  \
0                      49197                     14906                  769   

   Вакансий с указанной зарплатой  Доля вакансий с зарплатой (%)  


In [39]:
query_7_3 = f'''SELECT 
                    -- Python статистика
                    COUNT(CASE WHEN key_skills ILIKE '%python%' THEN 1 END) AS "Вакансий с Python",
                    COUNT(DISTINCT CASE WHEN key_skills ILIKE '%python%' THEN employer_id END) AS "Работодателей с Python",
                    
                    -- PostgreSQL статистика
                    COUNT(CASE WHEN key_skills ILIKE '%postgres%' OR key_skills ILIKE '%postgre%' THEN 1 END) AS "Вакансий с PostgreSQL",
                    COUNT(DISTINCT CASE WHEN key_skills ILIKE '%postgres%' OR key_skills ILIKE '%postgre%' THEN employer_id END) AS "Работодателей с PostgreSQL",
                    
                    -- Общая статистика
                    COUNT(*) AS "Всего вакансий",
                    COUNT(DISTINCT employer_id) AS "Всего работодателей"
                FROM public.vacancies;
'''
print('')
print('Популярность python и posgre:')
df = pd.read_sql_query(query_7_3, connection)
print(df)


Популярность python и posgre:
   Вакансий с Python  Работодателей с Python  Вакансий с PostgreSQL  \
0               3461                    1403                   2496   

   Работодателей с PostgreSQL  Всего вакансий  Всего работодателей  
0                        1175           49197                14906  


In [ ]:
query_7_4 = f'''WITH vacancy_salary AS (
                    -- Базовый CTE с вычисленной зарплатой
                    SELECT 
                        id,
                        experience,
                        key_skills,
                        COALESCE(
                            (salary_from + salary_to) / 2,
                            salary_from,
                            salary_to
                        ) AS salary
                    FROM public.vacancies
                    WHERE salary_from IS NOT NULL OR salary_to IS NOT NULL
                ),
                -- CTE для статистики по технологиям с разбивкой по опыту
                tech_by_experience AS (
                    -- Python
                    SELECT 
                        'Python' AS technology,
                        experience,
                        COUNT(*) AS vacancies_count,
                        ROUND(AVG(salary), 0) AS avg_salary
                    FROM vacancy_salary
                    WHERE key_skills ILIKE '%python%'
                    GROUP BY experience
                    
                    UNION ALL
                    
                    -- PostgreSQL
                    SELECT 
                        'PostgreSQL' AS technology,
                        experience,
                        COUNT(*) AS vacancies_count,
                        ROUND(AVG(salary), 0) AS avg_salary
                    FROM vacancy_salary
                    WHERE key_skills ILIKE '%postgres%' OR key_skills ILIKE '%postgre%'
                    GROUP BY experience
                    
                    UNION ALL
                    
                    -- SQL
                    SELECT 
                        'SQL' AS technology,
                        experience,
                        COUNT(*) AS vacancies_count,
                        ROUND(AVG(salary), 0) AS avg_salary
                    FROM vacancy_salary
                    WHERE key_skills ILIKE '%sql%'
                    GROUP BY experience
                ),
                -- CTE для сортировки опыта
                ordered_experience AS (
                    SELECT 
                        technology,
                        experience,
                        vacancies_count,
                        avg_salary,
                        CASE experience
                            WHEN 'Нет опыта' THEN 1
                            WHEN 'От 1 года до 3 лет' THEN 2
                            WHEN 'От 3 до 6 лет' THEN 3
                            WHEN 'Более 6 лет' THEN 4
                            ELSE 5
                        END AS exp_order
                    FROM tech_by_experience
                )
                -- Основной запрос
                SELECT 
                    technology AS "Технология",
                    experience AS "Опыт",
                    vacancies_count AS "Вакансий",
                    avg_salary AS "Средняя зарплата (₽)"
                FROM ordered_experience
                ORDER BY technology, exp_order;
'''
print('')
print('Популярность python и posgre, средняя зп:')
df = pd.read_sql_query(query_7_4, connection)
print(df)
connection.close()


Популярность python и posgre, средняя зп:
    Технология                Опыт  Вакансий  Средняя зарплата (₽)
0   PostgreSQL           Нет опыта        56               77606.0
1   PostgreSQL  От 1 года до 3 лет       440              117706.0
2   PostgreSQL       От 3 до 6 лет       433              189921.0
3   PostgreSQL         Более 6 лет        54              183159.0
4       Python           Нет опыта        86               68902.0
5       Python  От 1 года до 3 лет       449              108991.0
6       Python       От 3 до 6 лет       313              203300.0
7       Python         Более 6 лет        35              230806.0
8          SQL           Нет опыта       373               57553.0
9          SQL  От 1 года до 3 лет      2093              100133.0
10         SQL       От 3 до 6 лет      1276              162956.0
11         SQL         Более 6 лет       117              178878.0
